# Phase 3 — Database & Goals

**Goal:** Prove the SQLite schema, versioned goals, and all CRUD operations work correctly.

All tests use an **in-memory SQLite database** (`:memory:`) — nothing written to disk, safe to run repeatedly.

By the end of this notebook you'll have:
- All three ORM models (`Goal`, `Session`, `Thread`) tested
- Versioned goal updates confirmed (append-only, never overwrite)
- The **goal context dict** — exactly what gets passed to the LLM coach each session
- Thread lifecycle (open → resolved) working
- A simulated full week that exercises the whole schema

## Cell 1 — Install check

In [1]:
import importlib, sys

required = ["sqlalchemy", "pydantic"]
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  {pkg} — OK")
    except ImportError as e:
        print(f"  {pkg} — MISSING: {e}")

import sqlalchemy, pydantic
print(f"\nSQLAlchemy {sqlalchemy.__version__}")
print(f"Pydantic   {pydantic.__version__}")
print(f"Python     {sys.version.split()[0]}")

  sqlalchemy — OK
  pydantic — OK

SQLAlchemy 2.0.48
Pydantic   2.12.5
Python     3.12.13


## Cell 2 — SQLAlchemy engine (in-memory)

In [2]:
from sqlalchemy import create_engine, event
from sqlalchemy.orm import sessionmaker, DeclarativeBase

# In-memory SQLite — isolated, safe to run repeatedly.
# check_same_thread=False is needed when the same connection is used
# across async contexts (matches the production setting).
engine = create_engine(
    "sqlite:///:memory:",
    connect_args={"check_same_thread": False},
    echo=False,  # set True to see every SQL statement
)

# Enable WAL mode — matches production setup (concurrent reads during sessions)
@event.listens_for(engine, "connect")
def set_wal_mode(dbapi_conn, connection_record):
    dbapi_conn.execute("PRAGMA journal_mode=WAL")

SessionLocal = sessionmaker(bind=engine, autocommit=False, autoflush=False)


class Base(DeclarativeBase):
    pass


print("Engine ready — sqlite:///:memory:")

Engine ready — sqlite:///:memory:


## Cell 3 — ORM models

In [3]:
from sqlalchemy import Column, Integer, String, Boolean, DateTime, Date, Text
from datetime import datetime, date


class Goal(Base):
    """
    Versioned goals — never overwrite an existing row.
    When a goal is updated, the old row gets active=False and a new row
    is inserted. Query WHERE active=True to get the current state.

    horizon: weekly | monthly | custom
    - weekly  → week_start is the ISO Monday of that week
    - monthly → month is 'YYYY-MM'
    - custom  → target_date is the deadline
    """
    __tablename__ = "goals"

    id          = Column(Integer, primary_key=True, autoincrement=True)
    horizon     = Column(String, nullable=False)             # weekly / monthly / custom
    text        = Column(Text, nullable=False)               # the goal text
    target_date = Column(Date, nullable=True)                # custom goals
    week_start  = Column(Date, nullable=True)                # weekly goals (ISO Monday)
    month       = Column(String, nullable=True)              # monthly goals (YYYY-MM)
    created_at  = Column(DateTime, nullable=False, default=datetime.utcnow)
    active      = Column(Boolean, nullable=False, default=True)
    progress    = Column(String, nullable=False, default="on_track")
    # progress values: on_track | at_risk | achieved | carried_forward

    def __repr__(self):
        status = "ACTIVE" if self.active else "archived"
        return f"<Goal id={self.id} [{self.horizon}] [{status}] [{self.progress}] {self.text[:50]!r}>"


class Session(Base):
    """
    One row per morning / evening / dropin interaction.
    transcript and summary are plain text; goal_signals and
    calendar_actions are JSON strings (loaded/dumped with json module).
    """
    __tablename__ = "sessions"

    id               = Column(Integer, primary_key=True, autoincrement=True)
    date             = Column(String, nullable=False)        # YYYY-MM-DD
    type             = Column(String, nullable=False)        # morning / evening / dropin
    transcript       = Column(Text, nullable=True)
    summary          = Column(Text, nullable=True)
    goal_signals     = Column(Text, nullable=True)           # JSON list
    calendar_actions = Column(Text, nullable=True)           # JSON list
    created_at       = Column(DateTime, nullable=False, default=datetime.utcnow)

    def __repr__(self):
        return f"<Session id={self.id} date={self.date} type={self.type}>"


class Thread(Base):
    """
    An unresolved topic that resurfaces across sessions.
    Examples: "Follow up with client on proposal", "Book dentist"
    Threads surface in the coach's context until resolved=True.
    """
    __tablename__ = "threads"

    id            = Column(Integer, primary_key=True, autoincrement=True)
    created_date  = Column(Date, nullable=False, default=date.today)
    description   = Column(Text, nullable=False)
    resolved      = Column(Boolean, nullable=False, default=False)
    last_surfaced = Column(Date, nullable=True)

    def __repr__(self):
        status = "resolved" if self.resolved else "open"
        return f"<Thread id={self.id} [{status}] {self.description[:50]!r}>"


print("Models defined: Goal, Session, Thread")

Models defined: Goal, Session, Thread


## Cell 4 — Create tables & Pydantic schemas

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, Optional
import json

# Create all tables in the in-memory DB
Base.metadata.create_all(engine)
print("Tables created:", list(Base.metadata.tables.keys()))


# ── Pydantic schemas ─────────────────────────────────────────────────────────
# These are used at the API boundary (FastAPI request/response) and
# for the goal context dict passed to the LLM.

class GoalProgress(BaseModel):
    """Signal extracted from a session — what moved, what stalled."""
    goal_id: int
    signal: Literal["moved", "stalled", "achieved", "not_discussed"]
    note: Optional[str] = None  # optional coach note


class GoalCreate(BaseModel):
    """Validated input when the user sets or updates a goal."""
    horizon: Literal["weekly", "monthly", "custom"]
    text: str = Field(..., min_length=3, max_length=500)
    target_date: Optional[date] = None   # custom only
    week_start: Optional[date] = None    # weekly only
    month: Optional[str] = None          # monthly only — "YYYY-MM"

    @field_validator("text")
    @classmethod
    def text_not_empty(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("goal text cannot be empty or whitespace")
        return v

    @field_validator("month")
    @classmethod
    def month_format(cls, v: Optional[str]) -> Optional[str]:
        if v is None:
            return v
        import re
        if not re.match(r"^\d{4}-\d{2}$", v):
            raise ValueError("month must be in YYYY-MM format")
        return v


class GoalRead(BaseModel):
    """Outbound — safe to serialise and send to LLM or frontend."""
    id: int
    horizon: str
    text: str
    target_date: Optional[date]
    week_start: Optional[date]
    month: Optional[str]
    created_at: datetime
    active: bool
    progress: str

    model_config = {"from_attributes": True}  # allows from_orm / model_validate


class ThreadCreate(BaseModel):
    description: str = Field(..., min_length=3, max_length=500)


class ThreadRead(BaseModel):
    id: int
    created_date: date
    description: str
    resolved: bool
    last_surfaced: Optional[date]

    model_config = {"from_attributes": True}


class SessionCreate(BaseModel):
    date: str = Field(..., pattern=r"^\d{4}-\d{2}-\d{2}$")
    type: Literal["morning", "evening", "dropin"]
    transcript: Optional[str] = None
    summary: Optional[str] = None
    goal_signals: Optional[list[GoalProgress]] = None
    calendar_actions: Optional[list[dict]] = None


print("Pydantic schemas ready: GoalCreate, GoalRead, GoalProgress, ThreadCreate, ThreadRead, SessionCreate")

Tables created: ['goals', 'sessions', 'threads']
Pydantic schemas ready: GoalCreate, GoalRead, GoalProgress, ThreadCreate, ThreadRead, SessionCreate


## Cell 5 — Goals CRUD functions

In [5]:
from sqlalchemy.orm import Session as DBSession


def create_goal(db: DBSession, payload: GoalCreate) -> Goal:
    """
    Insert a new active goal row.
    If a matching active goal already exists for the same horizon/period,
    this does NOT automatically deactivate it — call update_goal() instead.
    """
    goal = Goal(
        horizon=payload.horizon,
        text=payload.text,
        target_date=payload.target_date,
        week_start=payload.week_start,
        month=payload.month,
        created_at=datetime.utcnow(),
        active=True,
        progress="on_track",
    )
    db.add(goal)
    db.commit()
    db.refresh(goal)
    return goal


def update_goal(db: DBSession, old_goal_id: int, new_text: str) -> Goal:
    """
    Versioned update:
      1. Mark the old row active=False.
      2. Insert a new row (copies horizon/period fields from the old row).
    Returns the new (active) Goal row.
    """
    old = db.get(Goal, old_goal_id)
    if old is None:
        raise ValueError(f"Goal id={old_goal_id} not found")
    if not old.active:
        raise ValueError(f"Goal id={old_goal_id} is already inactive — update the active version")

    old.active = False
    db.flush()  # write without committing so the new row gets a later created_at

    new_goal = Goal(
        horizon=old.horizon,
        text=new_text.strip(),
        target_date=old.target_date,
        week_start=old.week_start,
        month=old.month,
        created_at=datetime.utcnow(),
        active=True,
        progress=old.progress,   # carry forward progress signal
    )
    db.add(new_goal)
    db.commit()
    db.refresh(new_goal)
    return new_goal


def get_active_goals(db: DBSession) -> list[Goal]:
    """All currently active goals across all horizons."""
    from sqlalchemy import select
    stmt = select(Goal).where(Goal.active == True).order_by(Goal.horizon, Goal.created_at)
    return list(db.execute(stmt).scalars())


def get_goal_history(db: DBSession, horizon: str, period_value: str | date) -> list[Goal]:
    """
    All versions of a goal for a given horizon + period, chronological.
    period_value:
      - weekly  → date object (the week_start Monday)
      - monthly → string 'YYYY-MM'
      - custom  → date object (target_date)
    """
    from sqlalchemy import select
    stmt = select(Goal).where(Goal.horizon == horizon)
    if horizon == "weekly":
        stmt = stmt.where(Goal.week_start == period_value)
    elif horizon == "monthly":
        stmt = stmt.where(Goal.month == str(period_value))
    elif horizon == "custom":
        stmt = stmt.where(Goal.target_date == period_value)
    stmt = stmt.order_by(Goal.created_at)
    return list(db.execute(stmt).scalars())


def update_goal_progress(db: DBSession, goal_id: int, progress: str) -> Goal:
    """Update progress signal on the active goal (in-place — not a new version)."""
    valid = {"on_track", "at_risk", "achieved", "carried_forward"}
    if progress not in valid:
        raise ValueError(f"progress must be one of {valid}")
    goal = db.get(Goal, goal_id)
    if goal is None:
        raise ValueError(f"Goal id={goal_id} not found")
    goal.progress = progress
    db.commit()
    db.refresh(goal)
    return goal


def build_goal_context(active_goals: list[Goal]) -> dict:
    """
    Build the dict that gets injected into the LLM system prompt.
    Grouped by horizon so the coach can reference them naturally.
    """
    weekly, monthly, custom = [], [], []
    for g in active_goals:
        entry = {"id": g.id, "text": g.text, "progress": g.progress}
        if g.horizon == "weekly":
            entry["week_start"] = g.week_start.isoformat() if g.week_start else None
            weekly.append(entry)
        elif g.horizon == "monthly":
            entry["month"] = g.month
            monthly.append(entry)
        elif g.horizon == "custom":
            entry["target_date"] = g.target_date.isoformat() if g.target_date else None
            custom.append(entry)
    return {"weekly": weekly, "monthly": monthly, "custom": custom}


print("Goals CRUD ready: create_goal, update_goal, get_active_goals, get_goal_history, update_goal_progress, build_goal_context")

Goals CRUD ready: create_goal, update_goal, get_active_goals, get_goal_history, update_goal_progress, build_goal_context


## Cell 6 — Goals CRUD: create & update (versioning test)

### Mock tests, don't change any goals

In [6]:
db = SessionLocal()

# ── Weekly goal — week of 2026-04-07 (Monday) ────────────────────────────────
weekly_v1 = create_goal(db, GoalCreate(
    horizon="weekly",
    text="Close the Acme proposal and send follow-up by Friday",
    week_start=date(2026, 4, 13),
))
print(f"Created:  {weekly_v1}")

# Simulate a mid-week update — goal text refined
weekly_v2 = update_goal(db, weekly_v1.id, "Close Acme proposal, send follow-up, and prep Q2 deck outline")
print(f"Updated:  {weekly_v2}")

# Verify versioning: 2 rows exist, only 1 is active
from sqlalchemy import select
all_weekly = list(db.execute(select(Goal).where(Goal.horizon == "weekly")).scalars())
active_weekly = [g for g in all_weekly if g.active]

print(f"\nAll weekly rows:    {len(all_weekly)} (expected 2)")
print(f"Active weekly rows: {len(active_weekly)} (expected 1)")
assert len(all_weekly) == 2, "Should have 2 rows — original + updated"
assert len(active_weekly) == 1, "Only the updated version should be active"
assert active_weekly[0].id == weekly_v2.id
print("Versioning assertion PASSED")

# ── Monthly goal ──────────────────────────────────────────────────────────────
monthly = create_goal(db, GoalCreate(
    horizon="monthly",
    text="Increase monthly recurring revenue by 15% through 2 new client closures",
    month="2026-04",
))
print(f"\nCreated:  {monthly}")

# ── Custom goal with a deadline ───────────────────────────────────────────────
custom = create_goal(db, GoalCreate(
    horizon="custom",
    text="Finish investor pitch deck",
    target_date=date(2026, 4, 20),
))
print(f"Created:  {custom}")

db.close()

Created:  <Goal id=1 [weekly] [ACTIVE] [on_track] 'Close the Acme proposal and send follow-up by Frid'>
Updated:  <Goal id=2 [weekly] [ACTIVE] [on_track] 'Close Acme proposal, send follow-up, and prep Q2 d'>

All weekly rows:    2 (expected 2)
Active weekly rows: 1 (expected 1)
Versioning assertion PASSED

Created:  <Goal id=3 [monthly] [ACTIVE] [on_track] 'Increase monthly recurring revenue by 15% through '>
Created:  <Goal id=4 [custom] [ACTIVE] [on_track] 'Finish investor pitch deck'>


/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2274972359.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),
/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2274972359.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),


## Cell 7 — Goal context dict (what the LLM receives)

In [7]:
import json

db = SessionLocal()

active = get_active_goals(db)
print(f"Active goals: {len(active)}\n")
for g in active:
    print(f"  {g}")

context = build_goal_context(active)
print("\n── Goal context dict (injected into LLM system prompt) ──")
print(json.dumps(context, indent=2, default=str))

# Also validate the Pydantic round-trip
goal_reads = [GoalRead.model_validate(g) for g in active]
print(f"\nPydantic GoalRead validation: {len(goal_reads)} goals serialised cleanly")

db.close()

Active goals: 3

  <Goal id=4 [custom] [ACTIVE] [on_track] 'Finish investor pitch deck'>
  <Goal id=3 [monthly] [ACTIVE] [on_track] 'Increase monthly recurring revenue by 15% through '>
  <Goal id=2 [weekly] [ACTIVE] [on_track] 'Close Acme proposal, send follow-up, and prep Q2 d'>

── Goal context dict (injected into LLM system prompt) ──
{
  "weekly": [
    {
      "id": 2,
      "text": "Close Acme proposal, send follow-up, and prep Q2 deck outline",
      "progress": "on_track",
      "week_start": "2026-04-13"
    }
  ],
  "monthly": [
    {
      "id": 3,
      "text": "Increase monthly recurring revenue by 15% through 2 new client closures",
      "progress": "on_track",
      "month": "2026-04"
    }
  ],
  "custom": [
    {
      "id": 4,
      "text": "Finish investor pitch deck",
      "progress": "on_track",
      "target_date": "2026-04-20"
    }
  ]
}

Pydantic GoalRead validation: 3 goals serialised cleanly


## Cell 8 — Goal history query

In [8]:
db = SessionLocal()

history = get_goal_history(db, "weekly", date(2026, 4, 13))
print(f"Version history for week of 2026-04-07 — {len(history)} version(s):\n")
for i, g in enumerate(history, 1):
    print(f"  v{i}  {g}")
    print(f"       text: {g.text!r}")
    print(f"       created_at: {g.created_at}  active: {g.active}\n")

assert len(history) == 2
assert history[0].active == False  # original
assert history[1].active == True   # updated
print("History assertions PASSED — append-only versioning confirmed")

db.close()

Version history for week of 2026-04-07 — 2 version(s):

  v1  <Goal id=1 [weekly] [archived] [on_track] 'Close the Acme proposal and send follow-up by Frid'>
       text: 'Close the Acme proposal and send follow-up by Friday'
       created_at: 2026-04-12 14:58:00.857444  active: False

  v2  <Goal id=2 [weekly] [ACTIVE] [on_track] 'Close Acme proposal, send follow-up, and prep Q2 d'>
       text: 'Close Acme proposal, send follow-up, and prep Q2 deck outline'
       created_at: 2026-04-12 14:58:00.860575  active: True

History assertions PASSED — append-only versioning confirmed


## Cell 9 — Session & Thread CRUD functions

In [9]:
from sqlalchemy import select


# ── Session CRUD ──────────────────────────────────────────────────────────────

def save_session(db: DBSession, payload: SessionCreate) -> Session:
    """Persist a completed session to SQLite."""
    session = Session(
        date=payload.date,
        type=payload.type,
        transcript=payload.transcript,
        summary=payload.summary,
        goal_signals=json.dumps(
            [gs.model_dump() for gs in payload.goal_signals]
            if payload.goal_signals else []
        ),
        calendar_actions=json.dumps(payload.calendar_actions or []),
        created_at=datetime.utcnow(),
    )
    db.add(session)
    db.commit()
    db.refresh(session)
    return session


def get_latest_session(db: DBSession, session_type: str | None = None) -> Session | None:
    """Most recent session, optionally filtered by type."""
    stmt = select(Session).order_by(Session.created_at.desc()).limit(1)
    if session_type:
        stmt = stmt.where(Session.type == session_type)
    return db.execute(stmt).scalar_one_or_none()


def get_sessions_for_date(db: DBSession, date_str: str) -> list[Session]:
    stmt = select(Session).where(Session.date == date_str).order_by(Session.created_at)
    return list(db.execute(stmt).scalars())


# ── Thread CRUD ───────────────────────────────────────────────────────────────

def create_thread(db: DBSession, description: str) -> Thread:
    t = Thread(
        created_date=date.today(),
        description=description.strip(),
        resolved=False,
    )
    db.add(t)
    db.commit()
    db.refresh(t)
    return t


def resolve_thread(db: DBSession, thread_id: int) -> Thread:
    t = db.get(Thread, thread_id)
    if t is None:
        raise ValueError(f"Thread id={thread_id} not found")
    t.resolved = True
    db.commit()
    db.refresh(t)
    return t


def get_open_threads(db: DBSession, max_age_days: int = 7) -> list[Thread]:
    """Open threads created within the last max_age_days days."""
    from datetime import timedelta
    cutoff = date.today() - timedelta(days=max_age_days)
    stmt = (
        select(Thread)
        .where(Thread.resolved == False)
        .where(Thread.created_date >= cutoff)
        .order_by(Thread.created_date)
    )
    return list(db.execute(stmt).scalars())


def surface_thread(db: DBSession, thread_id: int) -> Thread:
    """Mark that this thread was surfaced to the user today."""
    t = db.get(Thread, thread_id)
    if t is None:
        raise ValueError(f"Thread id={thread_id} not found")
    t.last_surfaced = date.today()
    db.commit()
    db.refresh(t)
    return t


print("Session & Thread CRUD ready: save_session, get_latest_session, get_sessions_for_date")
print("                             create_thread, resolve_thread, get_open_threads, surface_thread")

Session & Thread CRUD ready: save_session, get_latest_session, get_sessions_for_date
                             create_thread, resolve_thread, get_open_threads, surface_thread


## Cell 10 — Session CRUD test

In [10]:
db = SessionLocal()

# Retrieve the active goals we created earlier (needed for goal_signals)
active_goals = get_active_goals(db)
weekly_active = next(g for g in active_goals if g.horizon == "weekly")
custom_active = next(g for g in active_goals if g.horizon == "custom")

# Save an evening session
saved = save_session(db, SessionCreate(
    date="2026-04-11",
    type="evening",
    transcript=(
        "Coach: How was your day? Tell me what's on your mind.\n"
        "Me: Had a productive afternoon — finally pushed the Acme proposal over the line. "
        "Still haven't touched the pitch deck though.\n"
        "Coach: Great progress on Acme. What's blocking the pitch deck?\n"
        "Me: Just need a solid 2-hour block with no interruptions."
    ),
    summary=(
        "Today I made real progress on the weekly goal — the Acme proposal is done and "
        "sent. The pitch deck is still untouched though, and I need to carve out focused "
        "time tomorrow morning to start it."
    ),
    goal_signals=[
        GoalProgress(goal_id=weekly_active.id, signal="moved", note="Acme proposal closed"),
        GoalProgress(goal_id=custom_active.id, signal="stalled", note="No time spent"),
    ],
))
print(f"Saved session: {saved}")
print(f"  goal_signals JSON: {saved.goal_signals}")

# Retrieve and verify
retrieved = get_latest_session(db, session_type="evening")
assert retrieved is not None
assert retrieved.date == "2026-04-11"
assert retrieved.type == "evening"

# Round-trip the goal_signals JSON
signals_back = [GoalProgress(**s) for s in json.loads(retrieved.goal_signals)]
print(f"\nGoal signals round-trip: {len(signals_back)} signals")
for s in signals_back:
    print(f"  goal_id={s.goal_id}  signal={s.signal}  note={s.note!r}")

print("\nSession CRUD assertions PASSED")
db.close()

Saved session: <Session id=1 date=2026-04-11 type=evening>
  goal_signals JSON: [{"goal_id": 2, "signal": "moved", "note": "Acme proposal closed"}, {"goal_id": 4, "signal": "stalled", "note": "No time spent"}]

Goal signals round-trip: 2 signals
  goal_id=2  signal=moved  note='Acme proposal closed'
  goal_id=4  signal=stalled  note='No time spent'

Session CRUD assertions PASSED


/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2829050578.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),


## Cell 11 — Thread CRUD test

In [11]:
db = SessionLocal()

# Create two threads
t1 = create_thread(db, "Follow up with Acme on signed contract — expected by EOW")
t2 = create_thread(db, "Book dentist appointment before end of month")

print(f"Created: {t1}")
print(f"Created: {t2}")

# Surface thread 1 (coach mentioned it in today's session)
surface_thread(db, t1.id)
print(f"\nSurfaced t1: last_surfaced={db.get(Thread, t1.id).last_surfaced}")

# Query open threads
open_threads = get_open_threads(db, max_age_days=7)
print(f"\nOpen threads (last 7 days): {len(open_threads)}")
for t in open_threads:
    print(f"  {t}")

assert len(open_threads) == 2

# Resolve thread 1 — Acme followed up
resolved = resolve_thread(db, t1.id)
print(f"\nResolved: {resolved}")

# Re-query
open_threads_after = get_open_threads(db, max_age_days=7)
print(f"Open threads after resolve: {len(open_threads_after)} (expected 1)")
assert len(open_threads_after) == 1
assert open_threads_after[0].id == t2.id

print("\nThread lifecycle assertions PASSED")
db.close()

Created: <Thread id=1 [open] 'Follow up with Acme on signed contract — expected '>
Created: <Thread id=2 [open] 'Book dentist appointment before end of month'>

Surfaced t1: last_surfaced=2026-04-12

Open threads (last 7 days): 2
  <Thread id=1 [open] 'Follow up with Acme on signed contract — expected '>
  <Thread id=2 [open] 'Book dentist appointment before end of month'>

Resolved: <Thread id=1 [resolved] 'Follow up with Acme on signed contract — expected '>
Open threads after resolve: 1 (expected 1)

Thread lifecycle assertions PASSED


## Cell 12 — Pydantic validation edge cases

Confirm that bad input is rejected at the schema layer before it ever reaches the DB.

In [12]:
from pydantic import ValidationError

cases = [
    ("empty text",       lambda: GoalCreate(horizon="weekly", text="   ")),
    ("bad horizon",      lambda: GoalCreate(horizon="quarterly", text="Do thing")),
    ("bad month fmt",    lambda: GoalCreate(horizon="monthly", text="Do thing", month="April-2026")),
    ("bad progress",     lambda: GoalProgress(goal_id=1, signal="blocked")),
    ("bad date fmt",     lambda: SessionCreate(date="11-04-2026", type="evening")),
    ("bad session type", lambda: SessionCreate(date="2026-04-11", type="break")),
]

all_passed = True
for label, fn in cases:
    try:
        fn()
        print(f"  FAIL — {label!r}: expected ValidationError but none raised")
        all_passed = False
    except ValidationError as e:
        field = e.errors()[0]["loc"]
        msg   = e.errors()[0]["msg"]
        print(f"  PASS — {label!r}: rejected at {field} — {msg}")

print(f"\nAll validation cases: {'PASSED' if all_passed else 'SOME FAILED'}")

  PASS — 'empty text': rejected at ('text',) — Value error, goal text cannot be empty or whitespace
  PASS — 'bad horizon': rejected at ('horizon',) — Input should be 'weekly', 'monthly' or 'custom'
  PASS — 'bad month fmt': rejected at ('month',) — Value error, month must be in YYYY-MM format
  PASS — 'bad progress': rejected at ('signal',) — Input should be 'moved', 'stalled', 'achieved' or 'not_discussed'
  PASS — 'bad date fmt': rejected at ('date',) — String should match pattern '^\d{4}-\d{2}-\d{2}$'
  PASS — 'bad session type': rejected at ('type',) — Input should be 'morning', 'evening' or 'dropin'

All validation cases: PASSED


## Cell 13 — End-to-end test: simulate a full week

Monday → set goals → update Wednesday → signal progress Thursday evening → verify state is correct for Friday morning context.

In [13]:
print("=" * 60)
print("END-TO-END TEST — simulated week (2026-04-07 to 2026-04-11)")
print("=" * 60)

db = SessionLocal()

# ── Monday: set this week's goals ────────────────────────────────────────────
print("\n[MON 04-07] Setting weekly and monthly goals...")
w = create_goal(db, GoalCreate(
    horizon="weekly",
    text="Write and send Q2 outreach to 10 prospects",
    week_start=date(2026, 4, 7),
))
m = create_goal(db, GoalCreate(
    horizon="monthly",
    text="Close 3 new clients in April",
    month="2026-04",
))
c = create_goal(db, GoalCreate(
    horizon="custom",
    text="Complete investor pitch deck",
    target_date=date(2026, 4, 20),
))
print(f"  Created: {w}")
print(f"  Created: {m}")
print(f"  Created: {c}")

# ── Tuesday: open a thread ────────────────────────────────────────────────────
print("\n[TUE 04-08] Coach creates an open thread...")
thread = create_thread(db, "Schedule follow-up call with prospect Jaya — she asked to reconnect")
print(f"  Thread: {thread}")

# ── Wednesday: refine weekly goal text ───────────────────────────────────────
print("\n[WED 04-09] Updating weekly goal...")
w_v2 = update_goal(db, w.id, "Write and send Q2 outreach to 10 prospects + draft personalised intros")
print(f"  Updated: {w_v2}")

# ── Wednesday evening: save session, signal progress ─────────────────────────
print("\n[WED 04-09 evening] Saving evening session...")
eve = save_session(db, SessionCreate(
    date="2026-04-09",
    type="evening",
    transcript="Coach: What did you get done today?\nMe: Sent 6 outreach emails. Deck untouched.",
    summary="Made solid progress on outreach — 6 done, 4 to go. Pitch deck hasn't started yet.",
    goal_signals=[
        GoalProgress(goal_id=w_v2.id, signal="moved", note="6/10 outreach sent"),
        GoalProgress(goal_id=c.id,    signal="stalled", note="No time"),
    ],
))
print(f"  Session saved: {eve}")

# ── Thursday: resolve the thread (Jaya call scheduled) ───────────────────────
print("\n[THU 04-10] Resolving thread — Jaya call booked...")
resolve_thread(db, thread.id)
print(f"  Thread resolved: {db.get(Thread, thread.id)}")

# ── Thursday evening: signal more progress ───────────────────────────────────
print("\n[THU 04-10 evening] Saving evening session...")
eve2 = save_session(db, SessionCreate(
    date="2026-04-10",
    type="evening",
    transcript="Coach: How did today go?\nMe: Finished all 10 outreach emails! Pitch deck: did the outline.",
    summary="Weekly outreach goal is done. Pitch deck has an outline now — moving forward.",
    goal_signals=[
        GoalProgress(goal_id=w_v2.id, signal="achieved", note="10/10 sent"),
        GoalProgress(goal_id=c.id,    signal="moved",    note="Outline completed"),
    ],
))
# Update progress signal in goals table
update_goal_progress(db, w_v2.id, "achieved")
update_goal_progress(db, c.id, "on_track")
print(f"  Session saved: {eve2}")

# ── Friday morning: verify context is correct ────────────────────────────────
print("\n[FRI 04-11] Assembling Friday morning context...")

active_goals = get_active_goals(db)
context = build_goal_context(active_goals)
open_threads = get_open_threads(db, max_age_days=7)
last_evening = get_latest_session(db, session_type="evening")

print("\n  Active goals:")
for g in active_goals:
    print(f"    {g}")

print(f"\n  Open threads: {len(open_threads)} (expected 0 — all resolved)")
print(f"  Last evening summary: {last_evening.summary[:80]!r}...")

print("\n  Goal context dict for LLM:")
print(json.dumps(context, indent=4, default=str))

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(active_goals) == 6, f"Expected 6 active goals, got {len(active_goals)}"
achieved = next(g for g in active_goals if g.horizon == "weekly")
assert achieved.progress == "achieved", f"Weekly goal should be achieved"
assert len(open_threads) == 0, "All threads should be resolved"
assert last_evening.date == "2026-04-10"

# Verify weekly goal history shows both versions
history = get_goal_history(db, "weekly", date(2026, 4, 13))
assert len(history) == 2
assert history[0].active == False
assert history[1].active == True

print("\n" + "=" * 60)
print("ALL END-TO-END ASSERTIONS PASSED")
print("Schema, versioning, sessions, threads, and context dict are correct.")
print("=" * 60)

db.close()

END-TO-END TEST — simulated week (2026-04-07 to 2026-04-11)

[MON 04-07] Setting weekly and monthly goals...
  Created: <Goal id=5 [weekly] [ACTIVE] [on_track] 'Write and send Q2 outreach to 10 prospects'>
  Created: <Goal id=6 [monthly] [ACTIVE] [on_track] 'Close 3 new clients in April'>
  Created: <Goal id=7 [custom] [ACTIVE] [on_track] 'Complete investor pitch deck'>

[TUE 04-08] Coach creates an open thread...
  Thread: <Thread id=3 [open] 'Schedule follow-up call with prospect Jaya — she a'>

[WED 04-09] Updating weekly goal...
  Updated: <Goal id=8 [weekly] [ACTIVE] [on_track] 'Write and send Q2 outreach to 10 prospects + draft'>

[WED 04-09 evening] Saving evening session...
  Session saved: <Session id=2 date=2026-04-09 type=evening>

[THU 04-10] Resolving thread — Jaya call booked...
  Thread resolved: <Thread id=3 [resolved] 'Schedule follow-up call with prospect Jaya — she a'>

[THU 04-10 evening] Saving evening session...
  Session saved: <Session id=3 date=2026-04-10 type=

/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2274972359.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),
/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2274972359.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),
/var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/ipykernel_97644/2829050578.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow(),


AssertionError: Weekly goal should be achieved

In [18]:
active_goals

[<Goal id=4 [custom] [ACTIVE] [on_track] 'Finish investor pitch deck'>,
 <Goal id=7 [custom] [ACTIVE] [on_track] 'Complete investor pitch deck'>,
 <Goal id=11 [custom] [ACTIVE] [on_track] 'Complete investor pitch deck'>,
 <Goal id=3 [monthly] [ACTIVE] [on_track] 'Increase monthly recurring revenue by 15% through '>,
 <Goal id=6 [monthly] [ACTIVE] [on_track] 'Close 3 new clients in April'>,
 <Goal id=10 [monthly] [ACTIVE] [on_track] 'Close 3 new clients in April'>,
 <Goal id=2 [weekly] [ACTIVE] [on_track] 'Close Acme proposal, send follow-up, and prep Q2 d'>,
 <Goal id=8 [weekly] [ACTIVE] [achieved] 'Write and send Q2 outreach to 10 prospects + draft'>,
 <Goal id=12 [weekly] [ACTIVE] [achieved] 'Write and send Q2 outreach to 10 prospects + draft'>]